## Import package

In [3]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',100)

## Import Risk Segment PD Output

In [5]:
risk_df=pd.read_pickle('../data/risk_segmented_output.pkl')
risk_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'total_rec_late_fee',
 'last_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',

In [6]:
risk_df[['actual_default_flag','pd_score','risk_band','risk_score']].tail(10)

,actual_default_flag,pd_score,risk_band,risk_score
692153,0,0.468976,Medium,531.0
1816200,0,0.233568,Low,766.0
1270714,0,0.050490,Very Low,950.0
1709748,0,0.533128,High,467.0
596968,0,0.739552,Very High,260.0
1135476,1,0.222382,Very Low,778.0
1601713,1,0.950497,Very High,50.0
344609,0,0.146255,Very Low,854.0
874428,1,0.554675,High,445.0
1910437,1,0.368561,Medium,631.0


In [7]:
risk_df.shape

(40000, 124)

In [8]:
full_df=pd.read_pickle('../data/lending_club_feature_engineered.pkl')
full_df.shape

(1340973, 128)

In [9]:
full_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 

## Recovery columns for LGD 

In [14]:
recovery_cols = [
    "funded_amnt",
    "funded_amnt_clean",
    "total_rec_prncp",
    "recoveries",
    "collection_recovery_fee",
    "int_rate",
    "grade",
    "purpose"
]

available_recovery_cols = [cols for cols in recovery_cols if cols in full_df.columns]
print(available_recovery_cols)

risk_df = risk_df.join(full_df[available_recovery_cols], rsuffix="_orig")
print(risk_df.shape)

['funded_amnt', 'funded_amnt_clean', 'total_rec_prncp', 'recoveries', 'collection_recovery_fee', 'int_rate', 'grade', 'purpose']
(40000, 140)


In [15]:
risk_df[available_recovery_cols].head()

,funded_amnt,funded_amnt_clean,total_rec_prncp,recoveries,collection_recovery_fee,int_rate,grade,purpose
824713,12000,12000,5138.16,0.0,0.0,19.89,E,debt_consolidation
591446,20100,20100,20100.00,0.0,0.0,15.59,C,debt_consolidation
1789024,8000,8000,8000.00,0.0,0.0,11.14,B,credit_card
1084390,23325,23325,23325.00,0.0,0.0,12.69,C,credit_card
1807296,21250,21250,21250.00,0.0,0.0,22.95,F,debt_consolidation


## Creating EAD (Exposure At Default)

In [16]:
risk_df['ead'] = risk_df['funded_amnt_clean']

risk_df['ead'] = risk_df['ead'].fillna(risk_df['funded_amnt'])
risk_df['ead'] = risk_df['ead'].replace(0, np.nan)

risk_df['ead'].describe()

count    40000.000000
mean     14513.557500
std       8711.779018
min        500.000000
25%       8000.000000
50%      12000.000000
75%      20000.000000
max      40000.000000
Name: ead, dtype: float64

## Creating Realized Loss

In [18]:
risk_df['realized_loss'] = risk_df['ead'] - risk_df['total_rec_prncp'].fillna(0) - risk_df['recoveries'].fillna(0)

risk_df['realized_loss'] = risk_df['realized_loss'].clip(lower=0)

risk_df[['ead', 'total_rec_prncp', 'recoveries', 'realized_loss']].head()

,ead,total_rec_prncp,recoveries,realized_loss
824713,12000,5138.16,0.0,6861.84
591446,20100,20100.00,0.0,0.00
1789024,8000,8000.00,0.0,0.00
1084390,23325,23325.00,0.0,0.00
1807296,21250,21250.00,0.0,0.00


## Creating Realized LGD

In [21]:
risk_df['realized_lgd'] = risk_df['realized_loss']/risk_df['ead'] \
    .replace([np.inf,-np.inf], np.nan)\
    .fillna(0)\
    .clip(lower=0, upper=1)

risk_df['realized_lgd'].describe()

count    40000.000000
mean      2229.085226
std       5400.007656
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      40000.000000
Name: realized_lgd, dtype: float64

## Estimate LGD Assumption from Defaulted Borrowers

In [23]:
defaulted_loans = risk_df[risk_df['actual_default_flag'] == 1].copy()

global_lgd = defaulted_loans['realized_lgd'].mean()
print(f'Global LGD Assumption : {global_lgd}')

Global LGD Assumption : 10063.590089164785


In [27]:
lgd_by_grade = defaulted_loans.groupby('grade')\
    .agg(
        defaulted_loan_count = ('actual_default_flag','count'),
        avg_realized_lgd = ('realized_lgd','mean'),
        avg_realized_loss = ('realized_loss','mean'),
        avg_ead = ('ead','mean')
    ).reset_index()

lgd_by_grade

,grade,defaulted_loan_count,avg_realized_lgd,avg_realized_loss,avg_ead
0,A,460,7011.446304,7011.446304,13186.793478
1,B,1704,7761.744126,7761.744126,13588.292254
2,C,2924,9378.598331,9378.598331,14892.886457
3,D,1997,10760.132439,10760.132439,16226.952929
4,E,1159,12972.271803,12972.271803,18428.515962
5,F,439,13736.755080,13736.755080,18962.015945
6,G,177,15456.646328,15456.646328,20283.192090


In [36]:
risk_df=risk_df.merge(lgd_by_grade[['grade','avg_realized_lgd']], on='grade', how='left')

risk_df['lgd_estimate'] = risk_df['avg_realized_lgd'].fillna(global_lgd)

risk_df[['grade','realized_lgd', 'lgd_estimate']].tail()

,grade,realized_lgd,lgd_estimate
39995,A,2430.54,7011.446304
39996,D,9424.90,10760.132439
39997,A,0.00,7011.446304
39998,C,2466.45,9378.598331
39999,D,4362.85,10760.132439


## Calculate Expected Loss

#### Expected Loss = PD * LGD * EAD

In [38]:
risk_df['expected_loss'] = risk_df['pd_score'] * risk_df['lgd_estimate'] * risk_df['ead']

risk_df[['pd_score', 'lgd_estimate', 'ead', 'expected_loss']].head()

,pd_score,lgd_estimate,ead,expected_loss
0,0.495415,12972.271803,12000,7.711987e+07
1,0.504460,9378.598331,20100,9.509568e+07
2,0.301881,7761.744126,8000,1.874498e+07
3,0.685799,9378.598331,23325,1.500226e+08
4,0.739917,13736.755080,21250,2.159862e+08


In [51]:
risk_df['interest_rate_decimal'] = risk_df['int_rate']/100
risk_df['expected_interest_income'] = risk_df['ead'] * risk_df['interest_rate_decimal']
risk_df['risk_adjusted_return'] = risk_df['expected_interest_income'] - risk_df['expected_loss']

risk_df[['int_rate', 'ead', 'expected_interest_income', 'risk_adjustment_return','expected_loss', 'expected_interest_income']].head()

,int_rate,ead,expected_interest_income,risk_adjustment_return,expected_loss,expected_interest_income
0,19.89,12000,2386.8000,-7.711748e+07,7.711987e+07,2386.8000
1,15.59,20100,3133.5900,-9.509254e+07,9.509568e+07,3133.5900
2,11.14,8000,891.2000,-1.874409e+07,1.874498e+07,891.2000
3,12.69,23325,2959.9425,-1.500196e+08,1.500226e+08,2959.9425
4,22.95,21250,4876.8750,-2.159813e+08,2.159862e+08,4876.8750


## Loss By Risk Band

In [54]:
loss_by_risk_band = risk_df.groupby('risk_band')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_risk_band

/var/folders/99/nqyz4s796bx09s6p71mvxd5h0000gn/T/ipykernel_3070/1092963251.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  loss_by_risk_band = risk_df.groupby('risk_band')\


,risk_band,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,Very Low,8000,0.048625,5.843970e+07,7304.961977,108462950,13557.868750,1.222017e+11,1.527521e+07,-1.221929e+11,-1.527411e+07
1,Low,8000,0.107500,6.543026e+07,8178.782460,101339225,12667.403125,2.448280e+11,3.060350e+07,-2.448167e+11,-3.060208e+07
2,Medium,8000,0.184125,7.251341e+07,9064.176279,108561100,13570.137500,4.174866e+11,5.218583e+07,-4.174721e+11,-5.218401e+07
3,High,8000,0.263875,7.993895e+07,9992.368650,122673975,15334.246875,6.984298e+11,8.730373e+07,-6.984105e+11,-8.730132e+07
4,Very High,8000,0.503375,8.896978e+07,11121.222968,139505050,17438.131250,1.194563e+12,1.493204e+08,-1.194537e+12,-1.493171e+08


## Loss By Purpose

In [55]:
loss_by_purpose = risk_df.groupby('purpose')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_purpose

,purpose,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,car,441,0.192744,3.841393e+06,8710.641761,3978100,9020.634921,1.670424e+10,3.787809e+07,-1.670372e+10,-3.787692e+07
1,credit_card,8738,0.189517,7.498660e+07,8581.666710,128972525,14759.959373,4.953421e+11,5.668827e+07,-4.953263e+11,-5.668647e+07
2,debt_consolidation,23248,0.232364,2.155470e+08,9271.635693,358433775,15417.832717,1.734339e+12,7.460164e+07,-1.734288e+12,-7.459945e+07
3,educational,10,0.100000,8.234894e+04,8234.894234,64100,6410.000000,1.379546e+08,1.379546e+07,-1.379474e+08,-1.379474e+07
4,home_improvement,2564,0.194228,2.300616e+07,8972.761489,36308925,14161.047192,1.545404e+11,6.027318e+07,-1.545355e+11,-6.027127e+07
5,house,212,0.231132,2.105307e+06,9930.692602,3284400,15492.452830,1.767001e+10,8.334911e+07,-1.766950e+10,-8.334672e+07
6,major_purchase,874,0.209382,7.783268e+06,8905.340393,10018450,11462.757437,4.693298e+10,5.369907e+07,-4.693161e+10,-5.369749e+07
7,medical,490,0.255102,4.616932e+06,9422.310713,4503725,9191.275510,2.222663e+10,4.536048e+07,-2.222598e+10,-4.535914e+07
8,moving,298,0.281879,2.901465e+06,9736.460572,2363725,7931.963087,1.235519e+10,4.146037e+07,-1.235482e+10,-4.145914e+07
9,other,2279,0.235630,2.200871e+07,9657.177719,22474075,9861.375603,1.181044e+11,5.182292e+07,-1.181010e+11,-5.182140e+07


## Loss By Grade

In [56]:
loss_by_grade = risk_df.groupby('grade')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_grade

,grade,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,A,6741,0.068239,4.726416e+07,7011.446304,93471175,13866.069574,1.162085e+11,1.723906e+07,-1.162019e+11,-1.723808e+07
1,B,11515,0.147981,8.937648e+07,7761.744126,151690400,13173.287017,4.143640e+11,3.598471e+07,-4.143478e+11,-3.598331e+07
2,C,11544,0.253292,1.082665e+08,9378.598331,165048100,14297.305960,7.973922e+11,6.907417e+07,-7.973690e+11,-6.907215e+07
3,D,6072,0.328887,6.533552e+07,10760.132439,94568300,15574.489460,6.329929e+11,1.042478e+08,-6.329761e+11,-1.042451e+08
4,E,2877,0.402850,3.732123e+07,12972.271803,51301550,17831.612791,4.601498e+11,1.599408e+08,-4.601389e+11,-1.599370e+08
5,F,935,0.469519,1.284387e+07,13736.755080,17717900,18949.625668,1.779099e+11,1.902780e+08,-1.779055e+11,-1.902732e+08
6,G,316,0.560127,4.884300e+06,15456.646328,6744875,21344.541139,7.849177e+10,2.483917e+08,-7.848989e+10,-2.483857e+08


## Save Outputs

In [ ]:
risk_df.to_pickle("../data/lgd_ead_expected_loss_output.pkl")

lgd_by_grade.to_csv("../data/lgd_by_grade.csv", index=False)
expected_loss_by_risk_band.to_csv("../data/expected_loss_by_risk_band.csv", index=False)
expected_loss_by_grade.to_csv("../data/expected_loss_by_grade.csv", index=False)
expected_loss_by_purpose.to_csv("../data/expected_loss_by_purpose.csv", index=False)

print("Saved LGD, EAD, and Expected Loss outputs.")